In [1]:
import joblib
import numpy as np

X_manual, y = joblib.load("../data/features_manual.pkl")
X_tfidf, y2, tfidf = joblib.load("../data/features_tfidf.pkl")
print(X_manual.shape, X_tfidf.shape)

(669639, 9) (669639, 20000)


In [2]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(y))
idx_train, idx_test = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)

y_train, y_test = y[idx_train], y[idx_test]
Xm_train, Xm_test = X_manual.iloc[idx_train], X_manual.iloc[idx_test]
Xt_train, Xt_test = X_tfidf[idx_train], X_tfidf[idx_test]
print(len(y_train), len(y_test))

535711 133928


In [3]:
from sklearn.metrics import classification_report, f1_score

results = {}

def evaluate(name, model, X_train, X_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    f1 = f1_score(y_test, pred, average="macro")
    results[name] = round(f1, 4)
    print(name, "→ F1 macro :", round(f1, 4))
    print(classification_report(y_test, pred))

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

evaluate("LogReg + manuelles",
         make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
         Xm_train, Xm_test)

LogReg + manuelles → F1 macro : 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     17940
           1       1.00      1.00      1.00     99360
           2       1.00      1.00      1.00     16628

    accuracy                           1.00    133928
   macro avg       1.00      1.00      1.00    133928
weighted avg       1.00      1.00      1.00    133928



In [6]:
from sklearn.ensemble import RandomForestClassifier

evaluate("RandomForest + manuelles",
         RandomForestClassifier(n_estimators=100, n_jobs=-1, class_weight="balanced", random_state=42),
         Xm_train, Xm_test)

RandomForest + manuelles → F1 macro : 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     17940
           1       1.00      1.00      1.00     99360
           2       1.00      1.00      1.00     16628

    accuracy                           1.00    133928
   macro avg       1.00      1.00      1.00    133928
weighted avg       1.00      1.00      1.00    133928



In [7]:
from xgboost import XGBClassifier

evaluate("XGBoost + manuelles",
         XGBClassifier(n_estimators=200, n_jobs=-1, random_state=42),
         Xm_train, Xm_test)

XGBoost + manuelles → F1 macro : 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     17940
           1       1.00      1.00      1.00     99360
           2       1.00      1.00      1.00     16628

    accuracy                           1.00    133928
   macro avg       1.00      1.00      1.00    133928
weighted avg       1.00      1.00      1.00    133928



In [8]:
evaluate("LogReg + TF-IDF",
         LogisticRegression(max_iter=300, class_weight="balanced"),
         Xt_train, Xt_test)

LogReg + TF-IDF → F1 macro : 0.886
              precision    recall  f1-score   support

           0       0.64      0.95      0.76     17940
           1       0.99      0.89      0.94     99360
           2       0.94      0.98      0.96     16628

    accuracy                           0.91    133928
   macro avg       0.85      0.94      0.89    133928
weighted avg       0.93      0.91      0.92    133928



In [9]:
import pandas as pd
pd.Series(results, name="F1 macro").sort_values(ascending=False)

LogReg + manuelles          1.000
RandomForest + manuelles    1.000
XGBoost + manuelles         1.000
LogReg + TF-IDF             0.886
Name: F1 macro, dtype: float64